In [0]:
from pyspark.sql.functions import col, to_date, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType


spark.conf.set(
    "fs.azure.account.key.<storage_account>.blob.core.windows.net", 
    "XNz5mbUfl1EEuLLsot9OVtAcu+lRIf5LmtdS8NrXzoHWFpy8bD/SMoMQaMdQ0w4A+BhVtnhkn4ZB+AStz/PMPw=="
)

raw_path = "wasbs://<contianer>@<storage_account>.blob.core.windows.net/raw/retail_dataset.csv"

bronze_path = "wasbs://<contianer>@<storage_account>.blob.core.windows.net/bronze"

# 2. Read the raw CSV
# Note: We use the exact columns from your file: order_id, customer_id, product, etc.
raw_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(raw_path)

# 3. Extract date components for partitioning (Requirement: year=2026/month=04)
from pyspark.sql.functions import year, month, col, to_date

# Convert the existing 'order_date' from your CSV to a date type
partitioned_df = raw_df.withColumn("order_date_dt", to_date(col("order_date"), "yyyy-MM-dd")) \
                       .withColumn("year", year(col("order_date_dt"))) \
                       .withColumn("month", month(col("order_date_dt")))

# 4. Write to Bronze layer using Partitioning
# This creates the /year=yyyy/month=mm/ folder structure automatically
partitioned_df.write.format("parquet") \
    .mode("append") \
    .partitionBy("year", "month") \
    .save(bronze_path)